# workflow

> Main workflow class for structure decomposition

In [ ]:
#| default_exp workflow.workflow

In [ ]:
#| export
from typing import Dict, Any, List, Optional
from fasthtml.common import Div, P, A, APIRouter
from fastcore.basics import patch

from cjm_fasthtml_interactions.patterns.step_flow import Step, StepFlow
from cjm_fasthtml_interactions.core.context import InteractionContext
from cjm_fasthtml_interactions.patterns.async_loading import AsyncLoadingContainer
from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui
from cjm_fasthtml_tailwind.utilities.spacing import m
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_plugin_system.core.manager import PluginManager
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

from cjm_fasthtml_workflow_transcript_decomp.core.config import StructureDecompWorkflowConfig
from cjm_transcript_source_select.models import SelectionUrls
from cjm_transcript_segmentation.models import SegmentationUrls, TextSegment
from cjm_transcript_vad_align.models import AlignmentUrls, VADChunk
from cjm_fasthtml_card_stack.core.models import CardStackUrls
from cjm_transcript_source_select.services.source import SourceService
from cjm_transcript_segmentation.services.segmentation import SegmentationService
from cjm_transcript_vad_align.services.alignment import AlignmentService
from cjm_transcript_review.services.graph import GraphService
from cjm_transcript_review.models import ReviewUrls
from cjm_transcript_review.components.review_card import AssembledSegment

# Step renderers
from cjm_transcript_source_select.components.step_renderer import render_selection_step
from cjm_fasthtml_workflow_transcript_decomp.combined.step_combined import render_combined_step
from cjm_transcript_review.components.step_renderer import render_review_step

## Session State Store Adapter

Bridges the pure `SQLiteWorkflowStateStore` (which accepts `session_id: str`)
to the `WorkflowStateStore` protocol expected by `StepFlow` (which passes `sess` objects).

In [ ]:
#| export
class _SessionStateStoreAdapter:
    """Adapter bridging StepFlow's sess-based protocol to the pure session_id-based store."""
    
    def __init__(
        self,
        store: SQLiteWorkflowStateStore  # The pure, framework-agnostic store
    ):
        """Wrap a pure store with session resolution."""
        self._store = store
    
    def get_current_step(
        self,
        flow_id: str,  # Workflow identifier
        sess: Any  # FastHTML session object
    ) -> Optional[str]:  # Current step ID or None
        """Get current step ID, resolving session from sess."""
        return self._store.get_current_step(flow_id, get_session_id(sess))
    
    def set_current_step(
        self,
        flow_id: str,  # Workflow identifier
        sess: Any,  # FastHTML session object
        step_id: str  # Step ID to set as current
    ) -> None:
        """Set current step ID, resolving session from sess."""
        self._store.set_current_step(flow_id, get_session_id(sess), step_id)
    
    def get_state(
        self,
        flow_id: str,  # Workflow identifier
        sess: Any  # FastHTML session object
    ) -> Dict[str, Any]:  # Workflow state dictionary
        """Get all workflow state, resolving session from sess."""
        return self._store.get_state(flow_id, get_session_id(sess))
    
    def update_state(
        self,
        flow_id: str,  # Workflow identifier
        sess: Any,  # FastHTML session object
        updates: Dict[str, Any]  # State updates to apply
    ) -> None:
        """Update workflow state, resolving session from sess."""
        self._store.update_state(flow_id, get_session_id(sess), updates)
    
    def clear_state(
        self,
        flow_id: str,  # Workflow identifier
        sess: Any  # FastHTML session object
    ) -> None:
        """Clear all workflow state, resolving session from sess."""
        self._store.clear_state(flow_id, get_session_id(sess))

## StructureDecompWorkflow

Main workflow class that orchestrates the structure decomposition process. It creates and manages:

- **SQLiteWorkflowStateStore**: Persistent state storage for workflow progress
- **Services**: Source, Segmentation, Alignment, and Graph services
- **StepFlow**: 3-step wizard with navigation
- **Routes**: API endpoints for the workflow

In [ ]:
#| export
class StructureDecompWorkflow:
    """Self-contained structure decomposition workflow."""
    
    # Class-level metadata for discovery
    workflow_name: str = "structure_decomposition"
    workflow_version: str = "1.0.0"
    workflow_category: str = "ingestion"
    workflow_title: str = "Structure Decomposition"
    workflow_description: str = "Decompose transcripts into traversable context graph spines"
    
    def __init__(
        self,
        plugin_manager: PluginManager,  # Plugin manager from host application
        config: Optional[StructureDecompWorkflowConfig] = None  # Workflow configuration
    ):
        """Initialize the workflow with injected PluginManager."""
        self.config = config or StructureDecompWorkflowConfig()
        self._app = None  # Set in setup()
        
        # Store the injected PluginManager
        self._plugin_manager = plugin_manager
        
        # Create SQLite-backed state store for persistence (pure, session_id-based)
        self._state_store = SQLiteWorkflowStateStore(
            db_path=self.config.get_state_db_path()
        )
        
        # Adapter for StepFlow compatibility (resolves sess -> session_id)
        self._session_adapter = _SessionStateStoreAdapter(self._state_store)
        
        # Create services
        self._source_service = SourceService(
            plugin_manager,
            source_categories=self.config.source_categories
        )
        self._segmentation_service = SegmentationService(
            plugin_manager,
            plugin_name=self.config.text_plugin
        )
        self._alignment_service = AlignmentService(
            plugin_manager,
            plugin_name=self.config.vad_plugin
        )
        self._graph_service = GraphService(
            plugin_manager,
            plugin_name=self.config.graph_plugin
        )
        
        # Create StepFlow
        self._step_flow = self._create_step_flow()
        
        # Create routers (focused routers for each domain)
        self._routers = self._create_routers()
        self._stepflow_router = self._step_flow.create_router(
            prefix=self.config.get_full_stepflow_prefix()
        )
    
    @classmethod
    def create_and_setup(
        cls,
        app,  # FastHTML application instance
        plugin_manager: PluginManager,  # Plugin manager from host application
        config: Optional[StructureDecompWorkflowConfig] = None  # Workflow configuration
    ) -> "StructureDecompWorkflow":  # Configured and setup workflow instance
        """Create, configure, and setup a workflow in one call."""
        workflow = cls(plugin_manager=plugin_manager, config=config)
        workflow.setup(app)
        return workflow
    
    @property
    def plugin_manager(self) -> PluginManager:  # Plugin manager instance
        """Access to plugin manager."""
        return self._plugin_manager
    
    @property
    def source_service(self) -> SourceService:  # Source service instance
        """Access to source service."""
        return self._source_service
    
    @property
    def segmentation_service(self) -> SegmentationService:  # Segmentation service instance
        """Access to segmentation service."""
        return self._segmentation_service
    
    @property
    def alignment_service(self) -> AlignmentService:  # Alignment service instance
        """Access to alignment service."""
        return self._alignment_service
    
    @property
    def graph_service(self) -> GraphService:  # Graph service instance
        """Access to graph service."""
        return self._graph_service
    
    @property
    def state_store(self) -> SQLiteWorkflowStateStore:  # State store instance
        """Access to state store."""
        return self._state_store
    
    @property
    def routers(self) -> List[APIRouter]:  # List of workflow routers
        """Access to workflow routers (excludes stepflow router)."""
        return self._routers
    
    @property
    def stepflow_router(self) -> APIRouter:  # StepFlow-generated router
        """StepFlow-generated router."""
        return self._stepflow_router

In [ ]:
#| export
@patch
def setup(
    self: StructureDecompWorkflow,
    app  # FastHTML application instance
) -> None:
    """Initialize workflow with FastHTML app."""
    self._app = app

In [ ]:
#| export
@patch
def cleanup(
    self: StructureDecompWorkflow
) -> None:
    """Clean up workflow resources."""
    self._app = None

In [ ]:
#| export
@patch
def get_routers(
    self: StructureDecompWorkflow
) -> List[APIRouter]:  # List of routers to register
    """Return all routers for registration with the app."""
    return self._routers + [self._stepflow_router]

In [ ]:
#| export
@patch
def render_entry_point(
    self: StructureDecompWorkflow,
    request,  # FastHTML request object
    sess  # FastHTML session object
):  # FastHTML component
    """Render the workflow entry point for embedding."""
    # Check if required plugins are available
    sources = self._source_service.get_available_sources()
    
    if not sources:
        # No source plugins available
        content = [
            P(
                "No transcription sources available. Please load transcription plugins first.",
                cls=combine_classes(text_dui.base_content.opacity(60), m.b(4))
            )
        ]
        if self.config.no_plugins_redirect:
            content.append(
                A(
                    "Go to Settings",
                    href=self.config.no_plugins_redirect,
                    cls=combine_classes(btn, btn_colors.primary)
                )
            )
        return Div(*content, id=self.config.container_id)
    
    # Sources available - load current status asynchronously
    current_status_url = self._core_routes["current_status"].to()
    return AsyncLoadingContainer(
        container_id=self.config.container_id,
        load_url=current_status_url
    )

In [ ]:
#| export
@patch
def _create_step_flow(
    self: StructureDecompWorkflow
) -> StepFlow:  # Configured StepFlow instance
    """Create and configure the StepFlow instance."""
    workflow = self
    
    # Data loaders for each step
    def load_sources(request) -> Dict[str, Any]:
        """Load available transcription sources."""
        sources = workflow._source_service.get_available_sources()
        transcriptions = workflow._source_service.query_transcriptions(limit=50)
        return {"sources": sources, "transcriptions": transcriptions}
    
    def load_segmentation_data(request) -> Dict[str, Any]:
        """Load data for segmentation step."""
        # Data is loaded via init route, not data_loader
        return {}
    
    def load_review_data(request) -> Dict[str, Any]:
        """Load data for review step."""
        return {}
    
    # Validation functions
    def validate_selection(state: Dict[str, Any]) -> bool:
        """Validate that sources have been selected."""
        step_states = state.get("step_states", {})
        selection_state = step_states.get("selection", {})
        selected_sources = selection_state.get("selected_sources", [])
        return len(selected_sources) > 0
    
    def validate_segmentation(state: Dict[str, Any]) -> bool:
        """Validate that segmentation and alignment are complete (1:1)."""
        step_states = state.get("step_states", {})

        seg_state = step_states.get("segmentation", {})
        segments = seg_state.get("segments", [])

        alignment_state = step_states.get("alignment", {})
        vad_chunks = alignment_state.get("vad_chunks", [])

        # Must have segments, VAD chunks, and counts must match (1:1 alignment)
        return len(segments) > 0 and len(vad_chunks) > 0 and len(segments) == len(vad_chunks)
    
    def validate_review(state: Dict[str, Any]) -> bool:
        """Validate review step."""
        return True
    
    # Completion handler
    async def on_complete(state: Dict[str, Any], request):
        """Handle workflow completion - commit to graph."""
        step_states = state.get("step_states", {})
        
        # Extract segments from segmentation step
        seg_state = step_states.get("segmentation", {})
        segment_dicts = seg_state.get("segments", [])
        segments = [TextSegment.from_dict(s) for s in segment_dicts]
        
        # Extract VAD chunks from alignment step
        align_state = step_states.get("alignment", {})
        chunk_dicts = align_state.get("vad_chunks", [])
        vad_chunks = [VADChunk.from_dict(c) for c in chunk_dicts]
        
        # Get document title from review state (or default)
        review_state = step_states.get("review", {})
        document_title = review_state.get("document_title", "Untitled Document")
        
        # Check if graph service is available
        if not workflow._graph_service.is_available():
            return Div(
                P("Graph plugin not loaded. Document was not committed."),
                id=workflow.config.container_id
            )
        
        # Commit to graph
        try:
            result = await workflow._graph_service.commit_document_async(
                title=document_title,
                text_segments=segments,
                vad_chunks=vad_chunks,
                media_type="audio",
            )
            segment_count = len(result.get("segment_ids", []))
            
            # Placeholder completion UI (Phase 4: Visualization will replace this)
            return Div(
                P(f"Committed {segment_count} segments to context graph."),
                P("Phase 4: Graph Visualization coming soon.", 
                  cls=combine_classes(text_dui.base_content.opacity(60), m.t(2))),
                id=workflow.config.container_id
            )
        except Exception as e:
            return Div(
                P(f"Commit failed: {str(e)}"),
                id=workflow.config.container_id
            )
    
    # Create render wrapper for selection step with explicit parameters
    def render_selection_step_with_urls(ctx: InteractionContext):
        """Render selection step extracting state and passing explicit parameters."""
        # Extract data from context (populated by data_loader)
        sources = ctx.get_data("sources", [])
        transcriptions = ctx.get_data("transcriptions", [])
        
        # Extract step state
        step_state = ctx.state.get("step_states", {}).get("selection", {})
        selected_sources = step_state.get("selected_sources", [])
        grouping_mode = step_state.get("grouping_mode", "media_path")
        external_db_paths = step_state.get("external_db_paths", [])
        file_browser_state = step_state.get("file_browser_state", {})
        
        # Active tab from workflow state (not step state)
        active_tab = ctx.get("source_tab", "db")
        
        return render_selection_step(
            sources=sources,
            transcriptions=transcriptions,
            selected_sources=selected_sources,
            grouping_mode=grouping_mode,
            external_db_paths=external_db_paths,
            file_browser_state=file_browser_state,
            active_tab=active_tab,
            urls=getattr(workflow, '_selection_urls', SelectionUrls()),
        )
    
    # Create render wrapper for combined step that injects URL bundles
    def render_combined_step_with_urls(ctx: InteractionContext):
        """Render combined segment & align step with URL bundles from workflow."""
        return render_combined_step(
            ctx=ctx,
            seg_urls=getattr(workflow, '_seg_urls', SegmentationUrls()),
            align_urls=getattr(workflow, '_align_urls', AlignmentUrls()),
            switch_chrome_url=getattr(workflow, '_switch_chrome_url', ''),
        )
    
    # Create render wrapper for review step
    def render_review_step_with_urls(ctx: InteractionContext):
        """Render review step by assembling segments with VAD chunks."""
        step_states = ctx.state.get("step_states", {})
        
        # Extract segments from segmentation step
        seg_state = step_states.get("segmentation", {})
        segment_dicts = seg_state.get("segments", [])
        segments = [TextSegment.from_dict(s) for s in segment_dicts]
        
        # Extract VAD chunks from alignment step
        align_state = step_states.get("alignment", {})
        chunk_dicts = align_state.get("vad_chunks", [])
        vad_chunks = [VADChunk.from_dict(c) for c in chunk_dicts]
        media_path = align_state.get("media_path")
        
        # Assemble segments with their corresponding VAD chunks
        assembled = [
            AssembledSegment(segment=seg, vad_chunk=chunk)
            for seg, chunk in zip(segments, vad_chunks)
        ]
        
        # Extract review step state
        review_state = step_states.get("review", {})
        focused_index = review_state.get("focused_index", 0)
        visible_count = review_state.get("visible_count", 5)
        is_auto_mode = review_state.get("is_auto_mode", False)
        card_width = review_state.get("card_width", 50)
        playback_speed = review_state.get("playback_speed", 1.0)
        auto_navigate = review_state.get("auto_navigate", False)
        
        # Keyboard system is managed internally by the library
        return render_review_step(
            assembled=assembled,
            focused_index=focused_index,
            visible_count=visible_count,
            is_auto_mode=is_auto_mode,
            card_width=card_width,
            playback_speed=playback_speed,
            auto_navigate=auto_navigate,
            urls=getattr(workflow, '_review_urls', ReviewUrls()),
            media_path=media_path,
        )
    
    return StepFlow(
        debug=True,
        flow_id=self.config.workflow_id,
        state_store=self._session_adapter,
        container_id=self.config.container_id,
        steps=[
            Step(
                id="selection",
                title="Select Sources",
                render=render_selection_step_with_urls,
                validate=validate_selection,
                data_loader=load_sources,
                data_keys=["selected_sources"],
                show_back=False,
                show_cancel=True,
                next_button_text="Continue"
            ),
            Step(
                id="segmentation",
                title="Segment & Align",
                render=render_combined_step_with_urls,
                validate=validate_segmentation,
                data_loader=load_segmentation_data,
                data_keys=["segments"],
                show_back=True,
                show_cancel=True,
                next_button_text="Continue"
            ),
            Step(
                id="review",
                title="Review",
                render=render_review_step_with_urls,
                validate=validate_review,
                data_loader=load_review_data,
                data_keys=[],
                show_back=True,
                show_cancel=True,
                next_button_text="Commit to Graph"
            )
        ],
        on_complete=on_complete,
        show_progress=self.config.show_progress,
        wrap_in_form=True
    )

In [ ]:
#| export
@patch
def _create_routers(
    self: StructureDecompWorkflow
) -> List[APIRouter]:  # List of configured APIRouters
    """Create the workflow's API routers."""
    from cjm_fasthtml_workflow_transcript_decomp.routes.init import init_routers
    return init_routers(self)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()